# Week 10 — Clustering and recommendation

This notebook goes with the Week 10 page. It uses the same two datasets and the same settings as the class demonstration. Here you can change the code, run it again, and see what happens.

Each grey box is a **code cell**. Click a cell and press **Shift + Enter** to run it. Run the cells in order, from top to bottom.

**In Google Colab**, everything this notebook needs is already installed. A notebook opened from GitHub does not keep your changes, so choose **File → Save a copy in Drive** before you start. Colab also disconnects after a while if you leave it idle. If that happens, run the cells again from the top.

**In VS Code**, install the Python and Jupyter extensions, open this file, and choose a Python environment as the kernel. Install the libraries once, by running `python -m pip install numpy pandas scikit-learn matplotlib` in a terminal.

## 1. Set up

This week the models come from `sklearn.cluster`. There are no labels and no train/test split.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN

## 2. The data

`make_blobs` generates points around a few centres. It also returns the group each point came from, but we throw that away with `_`. Clustering only gets the points.

Fixing `random_state` means you get exactly the same 180 points every time, and the same points as the Week 10 page.

In [ ]:
X, _ = make_blobs(
    n_samples=180,
    centers=[(-3.0, -1.0), (0.0, 2.8), (3.0, -0.8)],
    cluster_std=[0.75, 0.80, 0.85],
    random_state=42,
)

print("X has shape", X.shape)
print("First five rows of X:")
print(X[:5])

There is no `y` this time. Each row of `X` is one point with two features, and nothing says which group it belongs to.

## 3. Look at the data

This cell defines a plotting function, which the rest of the notebook reuses. Without labels, every point is grey. When you pass it cluster labels, each cluster gets its own colour. Points labelled `-1` are drawn as black crosses. Black stars mark centroids.

In [ ]:
def plot_clusters(X, labels=None, centres=None, title=""):
    fig, ax = plt.subplots(figsize=(7, 5))
    if labels is None:
        ax.scatter(*X.T, s=30, color="grey", label="unlabelled")
    else:
        for label in sorted(set(labels)):
            points = X[labels == label]
            if label == -1:
                ax.scatter(*points.T, s=60, color="black", marker="x", label="noise")
            else:
                ax.scatter(*points.T, s=30, color=f"C{label}", label=f"cluster {label}")
    if centres is not None:
        ax.scatter(*centres.T, s=300, color="black", marker="*", edgecolors="white", label="centroids")
    ax.set(xlim=(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5),
           ylim=(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5),
           xlabel="Feature 1", ylabel="Feature 2", title=title)
    ax.set_aspect("equal")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)
    plt.show()

plot_clusters(X, title="180 unlabelled points")

**Predict before you run the next cell.** How many groups would you ask for? Where do you expect the centroids to end up?

## 4. K-means

K-means needs one decision from us before it starts. `n_clusters` is `k`, the number of clusters to request. `fit_predict` gives each point a cluster number. The fitted model also keeps the centroids in `cluster_centers_` and the inertia in `inertia_`.

In [ ]:
model = KMeans(n_clusters=3, random_state=42, n_init=10)
labels = model.fit_predict(X)

print("Cluster numbers of the first ten points:", labels[:10])
print("Points in each cluster:", np.bincount(labels))
print("Centroids:")
print(model.cluster_centers_.round(2))
print("Inertia:", round(model.inertia_, 1))
plot_clusters(X, labels, model.cluster_centers_, "K-means with k = 3")

The numbers 0, 1 and 2 are only identifiers. They are not class names, and they mean nothing until somebody interprets them. Change `random_state` and the same groups may get different numbers.

### Does `k` matter?

**Predict before you run the next cell.** What will K-means do with `k = 2`? And with `k = 5`? Which value will give the lowest inertia?

The loop fits one model for each value of `k`, plots it, and collects the cluster sizes and the inertia in a `pandas` table.

In [ ]:
rows = []
for k in [2, 3, 5]:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    plot_clusters(X, labels, model.cluster_centers_, f"K-means with k = {k}")
    rows.append({
        "k": k,
        "cluster sizes": np.bincount(labels).tolist(),
        "inertia": model.inertia_,
    })

pd.DataFrame(rows).round(1)

The data did not change. Only the number of groups we asked for did. The lowest inertia belongs to `k = 5`, the setting that splits two good regions in half.

### Inertia keeps falling

**Predict before you run the next cell.** What happens to inertia as `k` goes from 1 to 10?

In [ ]:
ks = range(1, 11)
inertias = [KMeans(n_clusters=k, random_state=42, n_init=10).fit(X).inertia_ for k in ks]

plt.plot(ks, inertias, marker="o")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title("Inertia for k = 1 to 10")
plt.show()

Inertia falls every time `k` rises. Choosing the lowest value would always mean choosing the largest `k`. The curve bends sharply at `k = 3`, but deciding where it bends is still a judgement.

### Try it yourself

1. Change the second centre in the `make_blobs` cell from `(0.0, 2.8)` to `(-1.5, 0.5)`, so two groups overlap. Run the cells again. What does `k = 3` find now?
2. Change `cluster_std` to `[1.5, 1.5, 1.5]`. Does the bend in the inertia curve still show?

In [ ]:
# Your experiments here

## 5. Groups that are not blobs

The second dataset has 220 points in two interlocking crescents. It is the same shape as in Week 9, with its own fixed settings.

In [ ]:
X_moons, _ = make_moons(n_samples=220, noise=0.08, random_state=7)

plot_clusters(X_moons, title="220 unlabelled points")

**Predict before you run the next cell.** K-means will give two clusters, because we ask for two. Will they follow the two crescents?

In [ ]:
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_moons)

print("Points in each cluster:", np.bincount(kmeans_labels))
plot_clusters(X_moons, kmeans_labels, kmeans.cluster_centers_, "K-means with k = 2")

K-means did not break. It assumes a group is whatever lies nearest a centre, and that does not describe a crescent.

### DBSCAN

DBSCAN looks for dense, connected regions of points. `eps` is the radius of the neighbourhood around each point. `min_samples` is how many points a neighbourhood needs before it counts as dense. DBSCAN is not told how many clusters to find. Points that belong to no dense region get the label `-1`, which means noise.

In [ ]:
dbscan = DBSCAN(eps=0.18, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_moons)

print("Clusters found:", len(set(dbscan_labels) - {-1}))
print("Noise points:", np.sum(dbscan_labels == -1))
plot_clusters(X_moons, dbscan_labels, title="DBSCAN with eps = 0.18")

DBSCAN found the two crescents without being told there were two. It also left four points out of every group. That is a valid result, not a failure.

The value `eps = 0.18` was tuned to the spacing of these crescents. It is not a general setting.

### Try it yourself

1. Run DBSCAN with `eps` set to 0.1 and then to 0.3. How many clusters and how many noise points do you get each time?
2. Keep `eps = 0.18` and change `min_samples` to 2 and then to 10. What changes?
3. Run DBSCAN on the blob data `X`. Can you find an `eps` that recovers the three groups?

In [ ]:
# Your experiments here

## 6. People who liked this also liked

The ratings table from the page goes into a `pandas` table. `np.nan` marks a rating that is missing.

In [ ]:
ratings = pd.DataFrame(
    {
        "Film A": [5, 5, 4, 2, 1],
        "Film B": [4, 5, 4, 1, 2],
        "Film C": [np.nan, 4, 5, 2, 1],
        "Film D": [1, 1, 2, 5, 4],
        "Film E": [np.nan, 2, 1, 4, 5],
    },
    index=["Aoife", "Ben", "Carla", "Darragh", "Eimear"],
)
ratings

**Predict before you run the next cell.** Which two people rate films most like Aoife?

To measure how alike two people are, we compare their ratings on the films Aoife has rated. The average size of the differences is small when two people agree.

In [ ]:
rated = ratings.columns[ratings.loc["Aoife"].notna()]
others = ratings.drop(index="Aoife")

difference = (others[rated] - ratings.loc["Aoife", rated]).abs().mean(axis=1)
difference.sort_values()

Ben and Carla are closest to Aoife. Their ratings for the two films Aoife has not seen are the evidence for a recommendation.

In [ ]:
closest = difference.nsmallest(2).index
ratings.loc[closest, ["Film C", "Film E"]].mean()

Film C scores much higher than Film E, so Film C is the recommendation. Nothing in the code knows what either film is about. This is **collaborative filtering**, which uses patterns in many people's ratings to estimate a rating that nobody has recorded yet.

### Try it yourself

1. Use only the single closest person instead of two. Does the recommendation change?
2. Add a new person to the table who likes Film E. Give them ratings close to Aoife's on Films A, B and D. What happens to the recommendation?

In [ ]:
# Your experiments here

## 7. What you practised

- Clustering points that have no labels, with K-means and DBSCAN from `sklearn.cluster`.
- Reading `labels_`, `cluster_centers_` and `inertia_` from a fitted model.
- Fitting the same model for several values of `k` and comparing the results in a `pandas` table.
- Seeing that the lowest inertia does not give the most useful clustering.
- Counting the noise points DBSCAN leaves out, which have the label `-1`.
- Working with missing values in a `pandas` table, and making a recommendation from similar people's ratings.